In [1]:
import os
import time
import google.generativeai as genai
from dotenv import load_dotenv
import json
import tiktoken

C:\Users\10610\AppData\Local\Temp\ipykernel_18680\278477248.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [5]:

MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-flash-latest")
load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise Exception("404 GEMINI_API_KEY n'éxiste pas dans .env")

genai.configure(api_key=API_KEY)

class GeminiSDK:
    def __init__(self, model_name=MODEL_NAME):
        self.model = genai.GenerativeModel(model_name)

    def generate(self, prompt):
        for attempt in range(3):
            try:
                t0 = time.time()
                response = self.model.generate_content(prompt)
                t1 = time.time()
                print(f"[TIME] Gemini generate_content: {t1 - t0:.3f}s")
                return response.text
            except Exception as exc:
                err = str(exc).lower()
                if "429" in err or "rate limit" in err or "504" in err: # reessayer 
                    print(f"[WARN] appel Gemini échoué ({attempt+1}/3) : {err}")
                    time.sleep(2 * (attempt + 1))
                    continue
                if "401" in err:
                    raise RuntimeError("Clé API invalide ou non autorisée") from exc
                raise
        return "Service temporairement indisponible, veuillez réessayer plus tard."

sdk = GeminiSDK()

def resume_neutre(neutre, max_items=3):
    sample = neutre[:max_items]
    if not sample:
        return ""
    return f"Neutre exemple : {', '.join(sample)}."

def construire_pref_utilisateur(records): #combiner les prompts
    aime = []
    neutre = []
    deteste = []
    for r in records:
        value = r.get("rating_value")
        if value == "aimé":
            aime.append(r["name_serie"])
        elif value == "neutre":
            neutre.append(r["name_serie"])
        elif value == "n’aime pas":
            deteste.append(r["name_serie"])

    parts = [
        f"Aime: [{', '.join(aime)}]" if aime else "",
        resume_neutre(neutre, max_items=3),
        f"N'aime pas: [{', '.join(deteste)}]" if deteste else ""
    ]
    return " ; ".join([p for p in parts if p])

def build_prompt_from_pref(pref_text, user_question):
    return (
        f"{pref_text}\n\n"
        f"Question : {user_question}\n")

def count_tokens(prompt):
    enc = tiktoken.get_encoding("cl100k_base")
    return len(enc.encode(prompt))

def recommendation_user(records, question_utilisateur):
    t0 = time.time()

    pref_text = construire_pref_utilisateur(records)
    prompt = build_prompt_from_pref(pref_text, question_utilisateur)

    token_count = count_tokens(prompt)
    print(f"[INFO] token_count={token_count}")

    result = sdk.generate(prompt)

    t1 = time.time()
    print(f"[TIME] total recommendation_user: {t1 - t0:.3f}s")
    return result

In [6]:
# 测试
records_demo = [
    {"id_user": 1, "name_serie": "肖申克的救赎", "rating_value": "aimé"},
    {"id_user": 1, "name_serie": "教父", "rating_value": "aimé"},
    {"id_user": 1, "name_serie": "变形金刚5", "rating_value": "n’aime pas"},
    {"id_user": 1, "name_serie": "速度与激情", "rating_value": "n’aime pas"},
    {"id_user": 1, "name_serie": "泰坦尼克号", "rating_value": "neutre"},
]
print(recommendation_user(records_demo, "Veuillez me recommander cinq films les plus populaires."))

[INFO] token_count=75
[TIME] Gemini generate_content: 106.218s
[TIME] total recommendation_user: 106.218s
Basé sur vos préférences (vous appréciez les chefs-d'œuvre narratifs profonds et les drames classiques, mais vous n'aimez pas les blockbusters d'action répétitifs ou purement commerciaux), voici cinq recommandations de films mondialement acclamés qui devraient vous plaire :

1. **《辛德勒的名单》(La Liste de Schindler)**
   - **Pourquoi :** Tout comme *Les Évadés*, c'est un film d'une immense puissance émotionnelle qui traite de la résilience humaine et de la moralité dans des moments sombres. C'est un chef-d'œuvre incontesté du cinéma.

2. **《阿甘正传》(Forrest Gump)**
   - **Pourquoi :** Un pilier du cinéma populaire qui, contrairement aux films d'action que vous n'aimez pas, mise tout sur l'écriture, le développement des personnages et une narration riche traversant l'histoire américaine.

3. **《盗梦空间》(Inception)**
   - **Pourquoi :** Si vous cherchez un film "populaire" mais intelligent. Con